# Table 1
This notebook summarizes test performance for:
- `CNN`
- `Transformer`
- `XGBoost (2-mers)`
- `LogReg (2-mers)`
- `DEFT`



In [1]:
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd 

def _find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'conf').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root from current working directory.')

ROOT = _find_repo_root()
RESULTS_ROOT = ROOT / 'results'

DATASET_ORDER = ['polymerase', 'promoters', 'mpra_enhancers']
DATASET_LABELS = {
    'polymerase': 'Polymerase',
    'promoters': 'Promoters',
    'mpra_enhancers': 'MPRA/Enhancers',
}

BASELINE_SPECS = [
    {
        'name': 'DEFT',
        'files_by_dataset': {
            'polymerase': 'polymerase/results_deft_multiple_seeds.csv',
            'promoters': 'promoters/results_deft_multiple_seeds.csv',
            'mpra_enhancers': 'mpra_enhancers/results_deft_multiple_seeds.csv',
        },
        'selection': {'depth': 6.0},
    },
    {
        'name': 'CNN',
        'filename': 'LitConvolutionalNetwork.csv',
        'selection': None,
    },
    {
        'name': 'Transformer',
        'filename': 'LitTransformer.csv',
        'selection': None,
    },
    {
        'name': 'XGBoost (2-mers)',
        'filename': 'xgboost__kmer_count__k2.csv',
        'selection': {'sweep_param': 'max_depth', 'sweep_value': 6.0},
    },
    {
        'name': 'LogReg (2-mers)',
        'filename': 'logreg__kmer_count__k2.csv',
        'selection': {'sweep_param': 'C'},
    },
]


def resolve_results_path(dataset: str, spec: Dict) -> Path:
    if 'files_by_dataset' in spec:
        rel = spec['files_by_dataset'][dataset]
        return RESULTS_ROOT / rel

    if 'filename' in spec:
        return RESULTS_ROOT / dataset / 'baselines' / spec['filename']

    raise ValueError(f"Spec must define filename or files_by_dataset: {spec['name']}")


def load_results(dataset: str, spec: Dict) -> pd.DataFrame:
    path = resolve_results_path(dataset, spec)
    if not path.exists():
        raise FileNotFoundError(f'Missing results file: {path}')
    return pd.read_csv(path)


def filter_test_split(df: pd.DataFrame) -> pd.DataFrame:
    if 'split' in df.columns:
        return df[df['split'].astype(str).str.lower() == 'test'].copy()
    if 'isTrain' in df.columns:
        return df[df['isTrain'] == False].copy()
    raise ValueError('Could not infer test split column (expected split or isTrain).')


def apply_selection(df: pd.DataFrame, selection: Dict | None) -> pd.DataFrame:
    if selection is None:
        return df

    out = df.copy()
    for col, val in selection.items():
        if col not in out.columns:
            # Allow specs to include keys not present in some schemas.
            continue

        if isinstance(val, (int, float, np.integer, np.floating)):
            out = out[np.isclose(out[col].astype(float), float(val))]
        else:
            out = out[out[col] == val]

    return out


def mean_std(series: pd.Series) -> tuple[float, float]:
    return float(series.mean()), float(series.std(ddof=1))


In [2]:
records: List[Dict] = []

for spec in BASELINE_SPECS:
    for dataset in DATASET_ORDER:
        df = load_results(dataset, spec)
        df_test = filter_test_split(df)
        df_sel = apply_selection(df_test, spec['selection'])

        if len(df_sel) == 0:
            raise ValueError(
                f"No rows after selection for baseline={spec['name']} dataset={dataset}."
            )

        acc_mean, acc_std = mean_std(df_sel['accuracy'])
        auprc_mean, auprc_std = mean_std(df_sel['auprc'])

        records.append({
            'baseline': spec['name'],
            'dataset': dataset,
            'dataset_label': DATASET_LABELS[dataset],
            'n_rows': len(df_sel),
            'accuracy_mean': acc_mean,
            'accuracy_std': acc_std,
            'auprc_mean': auprc_mean,
            'auprc_std': auprc_std,
        })

summary_long = pd.DataFrame(records)
summary_long


,baseline,dataset,dataset_label,n_rows,accuracy_mean,accuracy_std,auprc_mean,auprc_std
0,DEFT,polymerase,Polymerase,5,0.869435,0.006589,0.912315,6.403795e-03
1,DEFT,promoters,Promoters,5,0.832566,0.003703,0.926355,3.536637e-03
2,DEFT,mpra_enhancers,MPRA/Enhancers,5,0.766667,0.048905,0.948105,1.208681e-02
3,CNN,polymerase,Polymerase,5,0.867834,0.007330,0.932746,5.752070e-03
4,CNN,promoters,Promoters,5,0.827718,0.004906,0.930871,1.883174e-03
5,CNN,mpra_enhancers,MPRA/Enhancers,5,0.790071,0.065214,0.963784,5.940968e-03
6,Transformer,polymerase,Polymerase,5,0.876138,0.016339,0.941811,1.992518e-03
7,Transformer,promoters,Promoters,5,0.835377,0.005858,0.933418,5.097986e-03
8,Transformer,mpra_enhancers,MPRA/Enhancers,5,0.703546,0.044616,0.952329,7.451913e-03
9,XGBoost (2-mers),polymerase,Polymerase,5,0.759180,0.005266,0.847530,2.327063e-03


In [3]:
def fmt_mean_std(mean: float, std: float, digits: int = 3) -> str:
    return f"\\meanstd{{{mean:.{digits}f}}}{{{std:.{digits}f}}}"


LATEX_LABELS = {
    'DEFT': r'\texttt{DEFT}',
    'CNN': r'\texttt{CNN}',
    'Transformer': r'\texttt{Transformer}',
    'XGBoost (2-mers)': r'\texttt{XGBoost}',
    'LogReg (2-mers)': r'\texttt{Log.} \texttt{Reg.}',
}

# Adjust this list to control row order in the final table.
LATEX_ROW_ORDER = [
    'DEFT',
    'CNN',
    'Transformer',
    'XGBoost (2-mers)',
    'LogReg (2-mers)',
]

# Add a \midrule before these rows.
MIDRULE_BEFORE = {'LogReg (2-mers)'}


def make_latex_row(summary: pd.DataFrame, baseline_name: str) -> str:
    label = LATEX_LABELS.get(baseline_name, baseline_name)
    row = [label]
    tmp = summary[summary['baseline'] == baseline_name].set_index('dataset')

    for ds in DATASET_ORDER:
        rec = tmp.loc[ds]
        row.append(fmt_mean_std(rec['accuracy_mean'], rec['accuracy_std']))
        row.append(fmt_mean_std(rec['auprc_mean'], rec['auprc_std']))

    return ' & '.join(row) + r' \\'


for baseline in LATEX_ROW_ORDER:
    if baseline not in set(summary_long['baseline']):
        continue
    if baseline in MIDRULE_BEFORE:
        print(r'\midrule')
    print(make_latex_row(summary_long, baseline))


\texttt{DEFT} & \meanstd{0.869}{0.007} & \meanstd{0.912}{0.006} & \meanstd{0.833}{0.004} & \meanstd{0.926}{0.004} & \meanstd{0.767}{0.049} & \meanstd{0.948}{0.012} \\
\texttt{CNN} & \meanstd{0.868}{0.007} & \meanstd{0.933}{0.006} & \meanstd{0.828}{0.005} & \meanstd{0.931}{0.002} & \meanstd{0.790}{0.065} & \meanstd{0.964}{0.006} \\
\texttt{Transformer} & \meanstd{0.876}{0.016} & \meanstd{0.942}{0.002} & \meanstd{0.835}{0.006} & \meanstd{0.933}{0.005} & \meanstd{0.704}{0.045} & \meanstd{0.952}{0.007} \\
\texttt{XGBoost} & \meanstd{0.759}{0.005} & \meanstd{0.848}{0.002} & \meanstd{0.822}{0.000} & \meanstd{0.925}{0.000} & \meanstd{0.777}{0.000} & \meanstd{0.965}{0.000} \\
\midrule
\texttt{Log.} \texttt{Reg.} & \meanstd{0.769}{0.002} & \meanstd{0.857}{0.001} & \meanstd{0.796}{0.000} & \meanstd{0.901}{0.000} & \meanstd{0.730}{0.000} & \meanstd{0.945}{0.000} \\


In [4]:
# Optional compact table for quick visual inspection
display_cols = []
for ds in DATASET_ORDER:
    display_cols.extend([f'{ds}_acc', f'{ds}_auprc'])

rows = []
for spec in BASELINE_SPECS:
    b = spec['name']
    rec = {'baseline': b}
    tmp = summary_long[summary_long['baseline'] == b].set_index('dataset')
    for ds in DATASET_ORDER:
        rec[f'{ds}_acc'] = f"{tmp.loc[ds, 'accuracy_mean']:.3f} ± {tmp.loc[ds, 'accuracy_std']:.3f}"
        rec[f'{ds}_auprc'] = f"{tmp.loc[ds, 'auprc_mean']:.3f} ± {tmp.loc[ds, 'auprc_std']:.3f}"
    rows.append(rec)

summary_wide = pd.DataFrame(rows)[['baseline', *display_cols]]
summary_wide


,baseline,polymerase_acc,polymerase_auprc,promoters_acc,promoters_auprc,mpra_enhancers_acc,mpra_enhancers_auprc
0,DEFT,0.869 ± 0.007,0.912 ± 0.006,0.833 ± 0.004,0.926 ± 0.004,0.767 ± 0.049,0.948 ± 0.012
1,CNN,0.868 ± 0.007,0.933 ± 0.006,0.828 ± 0.005,0.931 ± 0.002,0.790 ± 0.065,0.964 ± 0.006
2,Transformer,0.876 ± 0.016,0.942 ± 0.002,0.835 ± 0.006,0.933 ± 0.005,0.704 ± 0.045,0.952 ± 0.007
3,XGBoost (2-mers),0.759 ± 0.005,0.848 ± 0.002,0.822 ± 0.000,0.925 ± 0.000,0.777 ± 0.000,0.965 ± 0.000
4,LogReg (2-mers),0.769 ± 0.002,0.857 ± 0.001,0.796 ± 0.000,0.901 ± 0.000,0.730 ± 0.000,0.945 ± 0.000
